# IMLO_Open Assesment
Y3921738

## Initialisation

### Imports 

In [ ]:
import numpy
import random
import torch
from torch import nn
# import torch.nn.functional as F 
from torch.utils.data import DataLoader
import torch.optim.lr_scheduler as Scheduler
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data import ConcatDataset
import torchvision.models as models
import matplotlib.pyplot as plt
 

### Hyper Parameters

In [ ]:
image_size = 256
batch_size = 32
epochs = 50
learning_rate = 1e-5

sgd_momentum = 0.9
sgd_weight_decay = 1e-3
sgd_dampening = 0

adam_decay = 1e-3

drop_out = 0.2

patience = 0
factor = 0.75
threshold = 1e-1
cooldown = 0

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

save_path = "./flowers-102.pth"

## Data Preprocessing

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize(size=(image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=45),  # Randomly rotate the image by up to 15 degrees
    transforms.RandomHorizontalFlip(),      # Randomly flip the image horizontally with a probability of 0.5
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),  # Randomly adjust brightness, contrast, saturation, and hue
    transforms.Resize(size=(image_size, image_size)),
    transforms.ToTensor(),  # Convert the image to a PyTorch tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize the image using ImageNet statistics
])

validation_data = datasets.Flowers102(root="data", split="val", transform=test_transform, download=True)
test_data = datasets.Flowers102(root="data", split="test", transform=test_transform, download=True)

training_data_original = datasets.Flowers102(root="data", split="train", transform=test_transform, download=True)
training_data_aug = datasets.Flowers102(root="data", split="train", transform=train_transform)
training_data_concat = ConcatDataset([training_data_aug, training_data_original])

train_dataloader = DataLoader(training_data_concat, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)
validation_dataloader = DataLoader(validation_data, batch_size=batch_size)

In [ ]:
# import numpy as np

# # Choose a random sample from the dataset
# random_sample = random.choice(training_data)

# # Extract the image and label from the random sample
# image, label = random_sample

# # Convert the PIL image to a NumPy array
# image_np = np.array(image)

# # Plot the image
# plt.imshow(image_np)
# plt.title(f"Label: {label}")
# plt.axis('off')  # Hide axes
# plt.show()

## Neural Network

### Base Network Class

In [ ]:
class ConvNetwork(nn.Module):
    """
    A class that represents a convolutional neural network.
    ...
    Methods:
        forward (x): Takes the input x and computes the output from the network
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.conv_stack = nn.Sequential(
            # Conv layer 1
            nn.Conv2d(3, 6, 5),
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Conv layer 2
            nn.Conv2d(6, 16, 5),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.linear_relu_stack = nn.Sequential(
            # Linear layer 1
            nn.Linear((image_size - 12)*(image_size - 12), 512), # image size -12 squared, for the conv and pool (image_size - 12)*(image_size - 12)
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(drop_out),
            # Linear layer 2
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(drop_out),
            # Out layer
            nn.Linear(512, 102)
        )

    def forward(self, x):
        """
        Takes input x and returns the output of the network
        ...
        Parameters:
            x (Torch Tensor): the input
        Returns:
            x (Torch Tensor): output of network
        """
        x = self.conv_stack(x)
        x = self.flatten(x)
        out = self.linear_relu_stack(x)
        return out


### Extra convolutional layers

In [ ]:
class ConvNetworkExtraConv(nn.Module):
    """
    A class that represents a convolutional neural network.
    ...
    Methods:
        forward (x): Takes the input x and computes the output from the network
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.conv_stack = nn.Sequential(
            # Conv layer 1
            nn.Conv2d(3, 32, 5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # half dimensions
            # Conv layer 2
            nn.Conv2d(32, 64, 5, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # half dimensions

            # Conv layer 3
            nn.Conv2d(64, 128, 5, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # half dimensions
            # Conv layer 4
            nn.Conv2d(128, 256, 5, padding=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # half dimensions

        )
        self.linear_relu_stack = nn.Sequential(
            # Linear layer 1
            nn.Linear((image_size/16)*(image_size/16), 512), # image size -12 squared, for the conv and pool (image_size - 12)*(image_size - 12)
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(drop_out),
            # Linear layer 2
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(drop_out),
            # Out layer
            nn.Linear(512, 102)
        )

    def forward(self, x):
        """
        Takes input x and returns the output of the network
        ...
        Parameters:
            x (Torch Tensor): the input
        Returns:
            x (Torch Tensor): output of network
        """
        x = self.conv_stack(x)
        x = self.flatten(x)
        out = self.linear_relu_stack(x)
        return out


### Generate Instance

In [ ]:
classifier = ConvNetwork().to(device)
# classifier.load_state_dict(torch.load(save_path)) # This is optional and lets us load a saved model we have trained before
# print(classifier)

# Training

### Training Parameters

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# still look at new optimisers
optimizer1 = torch.optim.SGD(classifier.parameters(), lr=learning_rate, momentum=sgd_momentum, dampening=sgd_dampening, weight_decay=sgd_weight_decay)
optimizer2 = torch.optim.Adam(classifier.parameters(), lr=learning_rate, weight_decay=adam_decay)
optimizer = optimizer2

#, patience=patience, factor=factor, threshold=threshold, cooldown=cooldown
scheduler = Scheduler.ReduceLROnPlateau(optimizer=optimizer)

### Training, Validation and Testing Loops

In [ ]:
def train_loop(dataloader, classifier, loss_fn, optimizer):
    losses = 0
    classifier.train()
    batches_ran = 0

    for batch, (X, y) in enumerate(dataloader):
        y_pred = classifier(X.to(device))
        loss = loss_fn(y_pred, y.to(device))
        losses += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batches_ran += 1

    avg_loss = losses/batches_ran
    return avg_loss

# This can also act as the loop for testing if we load the testing data loader
def validation_loop(dataloader, classifier, loss_fn):
    classifier.eval()
    size = len(dataloader.dataset)
    loss = 0
    accuracy = 0

    with torch.no_grad():
        for (X, y) in dataloader:
            y_pred = classifier(X.to(device))
            loss += loss_fn(y_pred, y.to(device)).item()
            accuracy += (y_pred.argmax(1) == y.to(device)).type(torch.float).sum().item()

    loss /= len(dataloader)
    accuracy = (accuracy / size) * 100
    return accuracy, loss


### Run Classifier on Training and Validation Data

In [ ]:
train_losses, val_accuracies, val_losses = [], [], []
print(f"Using {device} device")
for i in range(epochs):
    train_loss = train_loop(train_dataloader, classifier, loss_fn, optimizer)
    val_accuracy, val_loss = validation_loop(validation_dataloader, classifier, loss_fn)
    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_accuracies.append(val_accuracy)
    val_losses.append(val_loss)
    print(f"Epoch {i+1} | Training Avg Loss: {train_loss:>8f} | Validation Avg Loss: {val_loss:>8f} | Validation Accuracy: {(val_accuracy):>0.3f}% | Learning rate: {scheduler.get_last_lr()}")
torch.cuda.synchronize()
print("Finished Training")

Using cuda device
Epoch 1 | Training Avg Loss: 4.574243 | Validation Avg Loss: 4.405978 | Validation Accuracy: 6.078% | Learning rate: [1e-05]
Epoch 2 | Training Avg Loss: 4.164081 | Validation Avg Loss: 4.278121 | Validation Accuracy: 9.510% | Learning rate: [1e-05]
Epoch 3 | Training Avg Loss: 3.927360 | Validation Avg Loss: 4.222047 | Validation Accuracy: 12.353% | Learning rate: [1e-05]
Epoch 4 | Training Avg Loss: 3.715869 | Validation Avg Loss: 4.169784 | Validation Accuracy: 15.000% | Learning rate: [1e-05]
Epoch 5 | Training Avg Loss: 3.542039 | Validation Avg Loss: 4.137594 | Validation Accuracy: 14.216% | Learning rate: [1e-05]
Epoch 6 | Training Avg Loss: 3.401927 | Validation Avg Loss: 4.109162 | Validation Accuracy: 15.686% | Learning rate: [1e-05]
Epoch 7 | Training Avg Loss: 3.254451 | Validation Avg Loss: 4.091740 | Validation Accuracy: 16.667% | Learning rate: [1e-05]
Epoch 8 | Training Avg Loss: 3.145354 | Validation Avg Loss: 4.057234 | Validation Accuracy: 17.647% |

# Testing And Evaluation

### Testing Data

In [ ]:
print("Testing:")
test_accuracy, test_loss = validation_loop(test_dataloader, classifier, loss_fn)
print(f"Loss: {test_loss} | Accuracy: {test_accuracy}")

### Graph Data

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, figsize=(12,6), sharex=True)
ax1.plot(range(len(train_losses)), train_losses, color="blue", label="Training Loss")
ax1.plot(range(len(val_losses)), val_losses, color="red", label="Validation Loss")
ax1.set(ylabel='Average Losses')
ax1.legend()
# ax2.axis([0, len(val_accuracies), 0, 100])
ax2.plot(range(len(val_accuracies)), val_accuracies, color="green", label="Validation Accuracy")
ax2.set(xlabel='Epochs', ylabel='Accuracy (%)')
ax2.legend()
print(f"Highest Accuracy: {(max(val_accuracies)):>0.2f}")

# Save the model

In [ ]:
# model = models.vgg16(weights='IMAGENET1K_V1')
# torch.save(model.state_dict(), save_path)